In [2]:
import csv
import re
import time
from pathlib import Path
from urllib.parse import urljoin
from tqdm import tqdm
import requests
from bs4 import BeautifulSoup

In [7]:
BASE = "https://support.bioconductor.org"
TAG_INDEX_URL = BASE + "/t/?page={page}"
OUT_CSV = Path("../02-datasource/bioconductor_all_tags.csv")

# The tag pages show:  × 58  → we extract 58
COUNT_RX = re.compile(r"[×x]\s*([\d,]+)")

HEADERS = {
    "User-Agent": "BioconductorTagCollector/1.0 (+research use; contact: you@example.com)"
}

def fetch(url: str) -> str:
    r = requests.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    return r.text

def parse_tag_page(html: str):
    """
    Yield (tag_name, count, href_abs) from one Bioconductor tag page.
    """
    soup = BeautifulSoup(html, "html.parser")

    # This container exists exactly in your HTML sample:
    container = soup.select_one("div.ui.seven.column.tag-list.grid")

    if not container:
        return

    items = container.select("div.item")

    for item in items:
        a = item.select_one("a.ptag[href]")
        if not a:
            continue

        tag = a.get_text(strip=True)
        href_abs = urljoin(BASE, a["href"])

        # Extract count "× 58"
        text = item.get_text(" ", strip=True)
        m = COUNT_RX.search(text)
        count = int(m.group(1).replace(",", "")) if m else 0

        yield tag, count, href_abs

def main():
    tag_counts = {}  # tag → count
    tag_href   = {}  # tag → url

    print("🔍 Detecting number of pages...")
    first_html = fetch(TAG_INDEX_URL.format(page=1))
    soup = BeautifulSoup(first_html, "html.parser")

    # Extract total pages from: "Page 5 of 213"
    pager_text = soup.get_text()
    m = re.search(r"Page\s+\d+\s+of\s+(\d+)", pager_text)
    MAX_PAGES = int(m.group(1)) if m else 1

    print(f"📄 Total pages detected: {MAX_PAGES}")

    for p in tqdm(range(1, MAX_PAGES + 1), desc="Scraping"):
        url = TAG_INDEX_URL.format(page=p)

        try:
            html = fetch(url)
        except Exception as e:
            print(f"[warn] failed page {p}: {e}")
            time.sleep(0.5)
            continue

        found = 0
        for tag, cnt, href in parse_tag_page(html):
            found += 1
            # Keep maximum count if seen multiple times
            prev = tag_counts.get(tag, 0)
            if cnt > prev:
                tag_counts[tag] = cnt
                tag_href[tag] = href

        time.sleep(0.3)
        # break

    # Save to CSV
    OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    with OUT_CSV.open("w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["tag", "count", "url"])
        for tag in sorted(tag_counts):
            w.writerow([tag, tag_counts[tag], tag_href[tag]])

    print(f"✅ Written: {OUT_CSV}  (total tags: {len(tag_counts)})")

if __name__ == "__main__":
    main()

🔍 Detecting number of pages...
📄 Total pages detected: 213


Scraping: 100%|██████████| 213/213 [03:59<00:00,  1.12s/it]

✅ Written: ../02-datasource/bioconductor_all_tags.csv  (total tags: 6513)


---

In [12]:
pip install matplotlib


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [10]:
import pandas as pd
import matplotlib.pyplot as plt

In [11]:
df = pd.read_csv('../02-datasource/bioconductor_all_tags.csv')
df.head()

,tag,count,url
0,"""",1,https://support.bioconductor.org/tag/%22/
1,"""bioconductor""",1,https://support.bioconductor.org/tag/%22biocon...
2,"""error:subscriptcontainsinvalidnames""",1,https://support.bioconductor.org/tag/%22error:...
3,"""genomeinfodbdata""package",1,https://support.bioconductor.org/tag/%22genome...
4,"""genomicranges"" ""r""",1,https://support.bioconductor.org/tag/%22genomi...


In [12]:
s = df['count'].dropna()
s.describe()

count    6.513000e+03
mean     1.230985e+04
std      9.924062e+05
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      3.000000e+00
max      8.009033e+07
Name: count, dtype: float64

In [13]:
df.count()

tag      6511
count    6513
url      6513
dtype: int64

In [14]:
df_filtered = df[df['count'] > 9]
s = df_filtered['count'].dropna()
s.describe()

count    8.140000e+02
mean     9.848137e+04
std      2.807161e+06
min      1.000000e+01
25%      1.500000e+01
50%      2.400000e+01
75%      5.700000e+01
max      8.009033e+07
Name: count, dtype: float64

In [15]:
df_filtered.count()

tag      814
count    814
url      814
dtype: int64

In [16]:
df_filtered.head(10)

,tag,count,url
114,450k,34,https://support.bioconductor.org/tag/450k/
125,8x60k,60,https://support.bioconductor.org/tag/8x60k/
135,a4,23,https://support.bioconductor.org/tag/a4/
143,abarray,12,https://support.bioconductor.org/tag/abarray/
155,acgh,77,https://support.bioconductor.org/tag/acgh/
171,adjusted pvalue,13,https://support.bioconductor.org/tag/adjusted%...
186,affxparser,36,https://support.bioconductor.org/tag/affxparser/
187,affy,1810,https://support.bioconductor.org/tag/affy/
192,affycomp,31,https://support.bioconductor.org/tag/affycomp/
194,affycoretools,49,https://support.bioconductor.org/tag/affycoret...


In [19]:
df_filtered.to_csv('../02-datasource/bioconductor_all_tags_[filtered].csv', index=False)

---

In [20]:
import json

tags = df_filtered["tag"].tolist()
with open("../02-datasource/bioconductor_tags.json", "w", encoding="utf-8") as f:
    json.dump(tags, f, indent=2, ensure_ascii=False)

In [22]:
with open("../02-datasource/bioconductor_tags_[filtered].json", "r", encoding="utf-8") as f:
    tag = json.load(f)

def remove_duplicates(lst):
    return list(dict.fromkeys(lst))

tag_filtered = remove_duplicates(tag)

In [23]:
len(tag_filtered)

43